In [1]:
!pip install transformers datasets seqeval -q


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
samples = [
    (["Alice", "is", "reading", "a", "book"], ["PROPN", "AUX", "VERB", "DET", "NOUN"]),
    (["They", "are", "playing", "football"], ["PRON", "AUX", "VERB", "NOUN"]),
    (["He", "bought", "a", "new", "car"], ["PRON", "VERB", "DET", "ADJ", "NOUN"]),
]

In [4]:
all_tags = sorted({tag for _, tags in samples for tag in tags})

tag2idx = {tag: idx for idx, tag in enumerate(all_tags)}
idx2tag = {idx: tag for tag, idx in tag2idx.items()}

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

c:\Users\dilip\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dilip\.cache\huggingface\hub\models--bert-base-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [7]:
def prepare_data(dataset):
    encoded = []

    for words, tags in dataset:
        tokenized = tokenizer(words, is_split_into_words=True, return_tensors="pt")
        word_ids = tokenized.word_ids()

        label_ids = []
        last_word = None

        for wid in word_ids:
            if wid is None:
                label_ids.append(-100)
            elif wid != last_word:
                label_ids.append(tag2idx[tags[wid]])
            else:
                label_ids.append(-100)

            last_word = wid

        encoded.append((tokenized, label_ids))

    return encoded


train_data = prepare_data(samples)

In [8]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(all_tags),
    id2label=idx2tag,
    label2id=tag2idx
)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
import torch

optimizer = torch.optim.Adam(model.parameters(), lr=4e-5)

model.train()

for ep in range(4):   # different epochs
    total_loss = 0

    for batch, labels in train_data:
        outputs = model(**batch, labels=torch.tensor([labels]))
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()

    print(f"Epoch {ep+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 6.1429
Epoch 2, Loss: 4.7219
Epoch 3, Loss: 3.6624
Epoch 4, Loss: 2.5973


In [10]:
def tag_sentence(sentence):
    words = sentence.split()
    inputs = tokenizer(words, return_tensors="pt", is_split_into_words=True)

    outputs = model(**inputs)
    preds = torch.argmax(outputs.logits, dim=2)[0]

    tagged = []
    for w, p in zip(words, preds):
        tagged.append((w, idx2tag[p.item()]))

    return tagged


print(tag_sentence("Alice bought a car"))

[('Alice', 'VERB'), ('bought', 'PROPN'), ('a', 'VERB'), ('car', 'DET')]


POS Tagging
Assigns syntactic roles to individual words
Helps identify grammatical structure
Chunking
Groups words into meaningful phrases
Slightly more complex than POS tagging